# Text Preprocessing

> 📘 **Python Mastery** · Module 16 — NLP · Lesson 2/5

Models eat numbers, not novels — preprocessing turns messy raw strings into consistent tokens so that `"GREAT!"`, `"great"`, and `"Great."` finally count as the same idea.

## 🎯 Learning Objectives

- Describe each stage of the classic text-preprocessing pipeline and why the order matters
- Clean text by stripping HTML tags, URLs, entities, and collapsed whitespace using regular expressions
- Tokenize with three different strategies and explain the tradeoffs (`str.split` vs `\w+` vs punctuation-aware)
- Expand contractions from a mapping and reason about **stopword removal** — including when it hurts
- Implement rule-based stemming and lookup-based lemmatization in pure Python and compare their artifacts
- Compose every stage into one reusable `preprocess()` function and inspect its effect on a real corpus

## 1. The Pipeline at a Glance

Nearly every classical NLP system pushes text through the same five gates. Think of it as food prep: wash, chop, measure — *then* cook.

| Stage | Question it answers | `"Didn't like it!"` becomes |
|---|---|---|
| 1. Clean | Is anything here *not* language? (HTML, URLs, noise) | `didn't like it` |
| 2. Tokenize | What counts as one unit? | `["didn't", "like", "it"]` |
| 3. Normalize | Are variants really different? (case, contractions) | `["did", "not", "like", "it"]` |
| 4. Stopwords | Which frequent glue-words can go? | `["not", "like"]` |
| 5. Stem / Lemma | Can forms collapse to one root? | `["not", "like"]` |

Order matters: removing stopwords before handling contractions would delete pieces of `didn't`; lowercasing before entity recognition destroys `Apple`. We will build each stage bottom-up, then assemble.

## 2. Stage 1 — Cleaning

Real text arrives wrapped in packaging: HTML tags from scraped pages, URLs pasted into reviews, entities like `&amp;`, and runs of stray whitespace. Regular expressions peel it off.

**Syntax:**

```python
import re, html

re.sub(r"<[^>]+>", " ", s)          # drop <tag> ... </tag>
re.sub(r"https?://\S+|www\.\S+", " ", s)   # drop URLs
html.unescape(s)                    # "&amp;" -> "&"
re.sub(r"\s+", " ", s).strip()      # collapse all whitespace runs
```

In [ ]:
import re
import html

scraped = "<p>Screen cracked within <b>two days</b>!</p> Details: http://shop.example.com/r42 &amp; photos."

step1 = re.sub(r"<[^>]+>", " ", scraped)                 # 1. strip tags
print(step1)

step2 = re.sub(r"https?://\S+|www\.\S+", " ", step1)     # 2. strip URLs
print(step2)

step3 = html.unescape(step2)                             # 3. decode entities
print(step3)

clean = re.sub(r"\s+", " ", step3).strip()               # 4. collapse whitespace
print(repr(clean))

### 2.1 Should You Lowercase Everything?

Lowercasing merges `Great`, `great`, `GREAT` into one token — usually exactly what bag-of-words models want. But casing sometimes *is* the information: `Apple` (company) vs `apple` (fruit), or an all-caps `NOT HAPPY` shouting frustration. Default to lowercasing for counting-style models; keep case when names or emotion matter (named-entity recognition, authorship, abuse detection).

**Example:**

In [ ]:
pair_a = "I work at Apple."
pair_b = "I eat an apple daily."

print(pair_a.lower())   # apple == apple  -> company sense lost
print(pair_b.lower())   # fine here

shout = "This is NOT what I ordered!!"
print(shout.isupper(), "<- casing carries emotion; lowercasing erases the shout")

# str.casefold(): like lower() but more aggressive (German 'ß' -> 'ss')
print("Straße".casefold(), "|", "STRASSE".casefold())
print("Straße".casefold() == "STRASSE".casefold())

## 3. Stage 2 — Tokenization

Tokenization decides what one "unit" is. Three escalating strategies, each fixing the previous one's blind spot:

| Strategy | Pattern | Blind spot |
|---|---|---|
| naive split | `text.split()` | leaves `"great!"` glued to its punctuation |
| plain word regex | `r"\w+"` | shreds `don't` into `don` + `t` |
| punctuation-aware | `r"\w+(?:'\w+)?"` plus standalone symbols | (our pick) |

**Syntax:**

```python
import re

text.split()                          # fastest, crudest
re.findall(r"\w+", text)              # words only, apostrophes cut
re.findall(r"\w+(?:'\w+)?|[^\w\s]", text)   # words (with ') + lone punctuation
```

In [ ]:
import re

sentence = "Don't panic: the refund arrived, and it's correct!"

print(sentence.split())                                # punctuation glued on
print(re.findall(r"\w+", sentence))                    # "Don", "t" torn apart
print(re.findall(r"\w+(?:'\w+)?|[^\w\s]", sentence))   # keeps don't / it's whole

## 4. Normalizing Contractions

English glues verbs to negations. Expanding `don't` → `do not` before any other step means later stages see honest words. A tiny dictionary handles most everyday cases — our map sends each token to a **list**, because one contraction can become several words.

In [ ]:
CONTRACTIONS = {
    "don't": ["do", "not"], "doesn't": ["does", "not"], "didn't": ["did", "not"],
    "can't": ["can", "not"], "cannot": ["can", "not"], "won't": ["will", "not"],
    "isn't": ["is", "not"], "aren't": ["are", "not"], "wasn't": ["was", "not"],
    "haven't": ["have", "not"], "hasn't": ["has", "not"], "hadn't": ["had", "not"],
    "couldn't": ["could", "not"], "shouldn't": ["should", "not"], "wouldn't": ["would", "not"],
    "it's": ["it", "is"], "i'm": ["i", "am"], "i've": ["i", "have"],
    "they're": ["they", "are"], "we're": ["we", "are"], "you're": ["you", "are"],
    "let's": ["let", "us"], "that's": ["that", "is"],
}

def expand(tokens):
    out = []
    for tok in tokens:
        out.extend(CONTRACTIONS.get(tok, [tok]))
    return out

demo = "don't worry: it's working, she wouldn't lie"
print(expand(demo.split()))

## 5. Stage 4 — Stopwords

**Stopwords** are high-frequency glue words (`the`, `of`, `is`) that appear in nearly every document and rarely change its topic. Removing them shrinks the vocabulary and sharpens the signal — most of the time. Below is a hand-authored English starter list (real systems ship lists of 100+ words).

In [ ]:
STOPWORDS = {
    "a", "an", "the", "and", "or", "but", "if", "then", "so", "because",
    "of", "at", "by", "for", "with", "about", "to", "from", "in", "on",
    "is", "am", "are", "was", "were", "be", "been", "being",
    "do", "does", "did", "have", "has", "had",
    "this", "that", "these", "those", "it", "its",
    "i", "you", "he", "she", "we", "they",
    "my", "your", "his", "her", "their", "our", "me", "him", "them",
    "not", "no", "never", "very", "just", "again", "only",
}
print(len(STOPWORDS), "stopwords loaded")

review = "the delivery was fast but the box was crushed"
kept = [w for w in review.split() if w not in STOPWORDS]
print(kept)

### 5.1 When Stopword Removal Hurts

Here is the classic trap. `not` sits in every standard stopword list — watch what removal does to sentiment:

In [ ]:
positive = "this phone is good"
negative = "this phone is not good"

strip_all = lambda s: [w for w in s.split() if w not in STOPWORDS]

print(strip_all(positive))
print(strip_all(negative))   # IDENTICAL to the positive sentence!

# Fix: keep negation words - they flip meaning, they are not glue.
NEGATION_KEEP = {"not", "no", "never"}
safe_stopwords = STOPWORDS - NEGATION_KEEP

strip_safe = lambda s: [w for w in s.split() if w not in safe_stopwords]
print(strip_safe(negative))

Rule of thumb: for *topic* tasks (search, clustering) drop stopwords freely; for *sentiment* or *meaning* tasks protect negations and contrast words like `but`.

## 6. Stage 5a — Stemming (Rules)

**Stemming** chops affixes with crude mechanical rules: fast, free, and occasionally violent. Our mini stemmer walks an ordered rule list — order matters, `("ies","y")` must fire before `("es","")`.

**Syntax:**

```python
STEM_RULES = [
    ("ational", "ate"), ("tional", "tion"),
    ("ities", ""), ("ity", ""),          # university -> univers
    ("ies", "y"),                        # studies -> study
    ("ingly", ""), ("ing", ""),          # playing -> play
    ("ied", "y"), ("ed", ""),            # played -> play
    ("ly", ""), ("ness", ""), ("ment", ""),
    ("es", ""), ("s", ""),               # scratches -> scratch
]

def mini_stem(word):
    for suffix, replacement in STEM_RULES:
        if word.endswith(suffix) and len(word) - len(suffix) >= 3:
            return word[: len(word) - len(suffix)] + replacement
    return word
```

In [ ]:
STEM_RULES = [
    ("ational", "ate"), ("tional", "tion"),
    ("ities", ""), ("ity", ""),
    ("ies", "y"),
    ("ingly", ""), ("ing", ""),
    ("ied", "y"), ("ed", ""),
    ("ly", ""), ("ness", ""), ("ment", ""),
    ("es", ""), ("s", ""),
]

def mini_stem(word):
    for suffix, replacement in STEM_RULES:
        if word.endswith(suffix) and len(word) - len(suffix) >= 3:
            return word[: len(word) - len(suffix)] + replacement
    return word

for w in ["playing", "played", "plays", "studies", "studied",
          "scratches", "crushed", "university", "universes"]:
    print(f"{w:>12} -> {mini_stem(w)}")

Read the output carefully — it shows both failure modes of stemming:

- **Overstemming:** `university` and `universes` both become `univers`, merging academia with outer space.
- **Understemming:** irregular pairs like `ran/run` share no suffix to chop, so they stay apart forever.
- Bonus artifact: `studied` → `studi` while `studying` → `study` — same verb, two different stems.

## 6b — Lemmatization (Lookup)

**Lemmatization** answers with a dictionary instead of a knife: map surface forms to real base words (*lemmas*). Pure-Python version = a lookup dict with the stemmer as fallback for regular forms.

In [ ]:
LEMMA_MAP = {
    "am": "be", "is": "be", "are": "be", "was": "be", "were": "be", "been": "be",
    "has": "have", "had": "have", "does": "do", "did": "do",
    "ran": "run", "felt": "feel", "went": "go", "bought": "buy", "arrived": "arrive",
    "mice": "mouse", "geese": "goose", "feet": "foot", "teeth": "tooth",
    "better": "good", "best": "good", "worse": "bad", "happily": "happy",
}

def lemmatize(word):
    return LEMMA_MAP.get(word, mini_stem(word))

import pandas as pd

words = ["studies", "studied", "running", "ran", "better",
         "mice", "arrived", "university", "happily", "crushed"]
pd.DataFrame({
    "original": words,
    "stemmed":  [mini_stem(w) for w in words],
    "lemmatized": [lemmatize(w) for w in words],
})

Notice `ran`: the stemmer cannot help, the lookup fixes it. Real lemmatizers also use part-of-speech context (`saw` as verb → `see`, as noun → `saw`) — dictionaries alone cannot do that; full pipelines like spaCy can.

## 7. Composing the Pipeline

Individually these steps are toys; together they are a function you would actually ship. One entry point, keyword flags for the risky decisions:

**Syntax:**

```python
def preprocess(text, remove_stops=True, keep_negations=True):
    text = re.sub(r"<[^>]+>", " ", text)                  # 1. clean
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"[^A-Za-z0-9' ]+", " ", text.lower())  # 2. normalize case/symbols
    tokens = re.findall(r"[a-z]+(?:'[a-z]+)?", text)      # 3. tokenize
    tokens = [t for tok in tokens                         # 4. expand contractions
              for t in CONTRACTIONS.get(tok, [tok])]
    stops = STOPWORDS - NEGATION_KEEP if (remove_stops and keep_negations) \
            else STOPWORDS if remove_stops else set()
    return [lemmatize(t) for t in tokens if t not in stops]   # 5. filter + lemmatize
```

In [ ]:
import re

CONTRACTIONS = {
    "don't": ["do", "not"], "doesn't": ["does", "not"], "didn't": ["did", "not"],
    "can't": ["can", "not"], "won't": ["will", "not"], "isn't": ["is", "not"],
    "aren't": ["are", "not"], "wasn't": ["was", "not"], "haven't": ["have", "not"],
    "hasn't": ["has", "not"], "couldn't": ["could", "not"], "wouldn't": ["would", "not"],
    "shouldn't": ["should", "not"], "it's": ["it", "is"], "i'm": ["i", "am"],
    "i've": ["i", "have"], "they're": ["they", "are"], "we're": ["we", "are"],
    "you're": ["you", "are"], "let's": ["let", "us"], "that's": ["that", "is"],
}
STOPWORDS = {
    "a", "an", "the", "and", "or", "but", "if", "then", "so", "because",
    "of", "at", "by", "for", "with", "about", "to", "from", "in", "on",
    "is", "am", "are", "was", "were", "be", "been", "being",
    "do", "does", "did", "have", "has", "had",
    "this", "that", "these", "those", "it", "its",
    "i", "you", "he", "she", "we", "they",
    "my", "your", "his", "her", "their", "our", "me", "him", "them",
    "not", "no", "never", "very", "just", "again", "only",
}
NEGATION_KEEP = {"not", "no", "never"}

STEM_RULES = [("ational", "ate"), ("tional", "tion"), ("ities", ""), ("ity", ""),
              ("ies", "y"), ("ingly", ""), ("ing", ""), ("ied", "y"), ("ed", ""),
              ("ly", ""), ("ness", ""), ("ment", ""), ("es", ""), ("s", "")]

def mini_stem(word):
    for suffix, replacement in STEM_RULES:
        if word.endswith(suffix) and len(word) - len(suffix) >= 3:
            return word[: len(word) - len(suffix)] + replacement
    return word

LEMMA_MAP = {
    "am": "be", "is": "be", "are": "be", "was": "be", "were": "be", "been": "be",
    "has": "have", "had": "have", "does": "do", "did": "do",
    "ran": "run", "felt": "feel", "went": "go", "bought": "buy", "arrived": "arrive",
    "mice": "mouse", "geese": "goose", "feet": "foot", "teeth": "tooth",
    "better": "good", "best": "good", "worse": "bad", "happily": "happy",
}

def lemmatize(word):
    return LEMMA_MAP.get(word, mini_stem(word))

def preprocess(text, remove_stops=True, keep_negations=True):
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"[^a-z0-9' ]+", " ", text.lower())
    tokens = re.findall(r"[a-z]+(?:'[a-z]+)?", text)
    tokens = [t for tok in tokens for t in CONTRACTIONS.get(tok, [tok])]
    if remove_stops:
        stops = STOPWORDS - NEGATION_KEEP if keep_negations else STOPWORDS
        tokens = [t for t in tokens if t not in stops]
    return [lemmatize(t) for t in tokens]

print(preprocess("<p>I DIDN'T like it!</p> Visit http://x.example.com"))
# ['like']  <- cleaned, expanded, filtered, lemmatised

Now run the whole pipeline over our module corpus (twelve authored reviews and support messages) and compare raw vs processed side by side:

In [ ]:
import pandas as pd

# same helper definitions as the previous cell live in memory now
CORPUS = [
    "I didn't expect much, but this keyboard is GREAT!",
    "It's been nine days and my order #7781 hasn't arrived.",
    "Track your parcel anytime at http://help.example.com/track.",
    "<p>Screen cracked within <b>two days</b>. Very unhappy.</p>",
    "The battery isn't terrible, but it won't survive a full day.",
    "Support can't be reached by phone and I'm stuck on hold again.",
    "Setup was easy and the app doesn't crash anymore.",
    "They refunded me in two days. Couldn't ask for better service.",
    "Why does the price change every week? It's confusing.",
    "The strap snapped during a run. Won't buy this brand again.",
    "<div>Sound quality is fine for podcasts, not for music.</div>",
    "Delivery was quick BUT the box was crushed. Mixed feelings.",
]

pd.set_option("display.max_colwidth", 60)
pd.DataFrame({
    "raw": CORPUS,
    "processed": [" ".join(preprocess(doc)) for doc in CORPUS],
})

> 🔍 **Under the Hood:** Python strings are immutable, so *every* pipeline stage above allocates a brand-new string object — `s.lower()` never edits `s` in place. For twelve documents that is irrelevant; for a 50 GB crawl it decides whether your job fits in RAM. That is why production pipelines stream documents lazily (generators) instead of materializing a giant list, and why heavy toolkits pre-tokenize once and cache the result.

In [ ]:
s = "Tea"
print(id(s))
print(id(s.lower()))    # different objects: a copy was made
big = "x" * 1_000_000
print(big.lower() is big)         # False: even trivial changes allocate...
print(big.upper().lower() == big) # ...while the CONTENT round-trips perfectly

## 8. Production Versions (NLTK & spaCy)

Our home-built pipeline teaches the concepts; production teams reach for battle-tested libraries. NLTK gives classic components, spaCy gives a fast integrated pipeline with linguistic annotations.

**NLTK** *(requires `pip install nltk` plus corpus downloads)*:

```bash
pip install nltk
python -m nltk.downloader punkt_tab stopwords wordnet omw-1.4
```

```python
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

tokens = word_tokenize("The batteries weren't included, sadly.")   # ['The','batteries','were', "n't", ...]
stops = set(stopwords.words("english"))                            # 179 English stopwords
lemma = WordNetLemmatizer()

[lemma.lemmatize(t.lower(), pos="n") for t in tokens if t.lower() not in stops]
# 'batteries' -> 'battery' - WordNet knows real dictionary forms
```

**spaCy** — one call returns tokens, lemmas, parts of speech, and entities:

```bash
pip install spacy
python -m spacy download en_core_web_sm       # small English model
```

```python
import spacy

nlp = spacy.load("en_core_web_sm")
doc = nlp("Apple is opening a new store in Dhaka next month.")

for token in doc:
    print(f"{token.text:>8} | {token.lemma_:>8} | {token.pos_}")
#   Apple |    apple |   PROPN     <- case kept, entity preserved
# opening |     open |    VERB

[(ent.text, ent.label_) for ent in doc.ents]   # [('Apple', 'ORG'), ('Dhaka', 'GPE')]
```

## ⚠️ Preprocessing Decisions Impact Downstream

Every knob below silently reshapes whatever model comes next:

| Decision | Helps | Hurts |
|---|---|---|
| Lowercasing | shrinks vocabulary, merges variants | loses `Apple` the company, erases SHOUTING |
| Stopword removal | smaller features, faster training | deletes `not` → sentiment models go blind |
| Aggressive stemming | collapses many forms cheaply | non-words (`studi`) break dictionary/embedding lookups |
| Lookup lemmatization | real words, readable features | misses words absent from the map |
| URL/tag stripping | removes noise tokens | deletes signal in spam/phishing detection |

## ⚠️ Common Mistakes & Gotchas

| Mistake | Problem | Fix |
|---|---|---|
| Dropping stopwords for sentiment tasks | `not` removed → "good" == "not good" | keep a negation whitelist |
| Preprocessing train and test differently | vectorizer meets unseen tokens, silent accuracy loss | wrap vectorizer + model in one `Pipeline` (next lesson) |
| Tokenizing with bare `.split()` | `"great!"` ≠ `"great"` doubles vocabulary | regex tokenizer with punctuation rules |
| Stemming before dictionary/embedding lookup | `studi` exists nowhere | lemmatize instead, or stem consistently everywhere |
| Cleaning *after* splitting into train/test stats | leaks corpus-level info | fit preprocessing choices on train only |

## 💡 Best Practices & Pro Tips

- Put the entire pipeline in ONE function (like `preprocess`) so it is testable, versionable, and applied identically everywhere.
- Make risky choices explicit parameters (`keep_negations=...`) — silent defaults cause silent bugs.
- Cache preprocessed text; re-running regex over gigabytes per epoch is wasted compute.
- **AI-engineering relevance:** preprocessing is part of the *model artifact*. If you redeploy a classifier but forget its exact cleaning rules, accuracy quietly decays — production teams version preprocessing code alongside model weights. scikit-learn's `Pipeline` (next lesson) exists precisely to enforce this pairing.

## 📌 Summary

| Stage / Tool | What it does | Example |
|---|---|---|
| `re.sub(r"<[^>]+>", " ", s)` | strip HTML | `"<p>hi</p>"` → `" hi "` |
| `re.sub(r"https?://\S+", " ", s)` | strip URLs | removes tracking links |
| `text.lower()` / `casefold()` | fold case | `"GREAT"` → `"great"` |
| `re.findall(r"\w+(?:'\w+)?", t)` | tokenize, keep `don't` | `["don't", "panic"]` |
| `CONTRACTIONS.get(tok, [tok])` | expand contractions | `don't` → `do not` |
| stopword filtering | drop glue words | keep `not` for sentiment! |
| `mini_stem(w)` | rule suffix stripping | `university` → `univers` (careful!) |
| `LEMMATIZE.get(w, ...)` | dictionary base form | `ran` → `run`, `better` → `good` |
| `preprocess(text)` | all stages composed | one call, consistent everywhere |

**Key takeaways**

- The pipeline order is load-bearing: clean → tokenize → expand → stopwords → stem/lemma.
- Every simplification trades information for consistency; choose per task, not per habit.
- Negations are the canonical example of "helpful defaults that destroy sentiment tasks".
- Whatever you build today must run identically at train time and predict time.

## 🔗 Next Lesson

Clean tokens still are not numbers — next we turn them into vectors: [03_Text_Representation_TFIDF](../03_Text_Representation_TFIDF/notes.ipynb) covers Bag-of-Words, TF-IDF, cosine similarity, and a spam-classifier mini project.